# Recommendation Modelling

Goal: recommend new Spotify tracks that are not already in the user's playlist.

This notebook builds a content-based recommender. It represents the user's playlist as a taste profile, represents candidate tracks with the same feature space, and ranks candidates by similarity to the playlist profile.

## 1. Setup

The recommender needs two datasets:

- `data/df.csv`: the user's existing playlist from the earlier pipeline.
- `data/candidate_tracks.csv`: songs that are not necessarily in the playlist and can be recommended.

If `candidate_tracks.csv` does not exist yet, the optional Spotify API section below can create one using the Search API.

In [ ]:
import ast
import os
import re
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

def find_data_dir():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        notebook_data = candidate / "notebooks" / "data"
        local_data = candidate / "data"
        if notebook_data.exists():
            return notebook_data
        if candidate.name == "notebooks" and local_data.exists():
            return local_data
    return Path("data")


DATA_DIR = find_data_dir()
DATA_PATH = DATA_DIR / "df.csv"
CANDIDATE_PATH = DATA_DIR / "candidate_tracks.csv"
OUTPUT_PATH = DATA_DIR / "recommendations.csv"


## 2. Load User Playlist

This is the user's known taste profile. Tracks from this playlist will be excluded from final recommendations.

In [ ]:
playlist_df = pd.read_csv(DATA_PATH)

playlist_df["source"] = "playlist"
playlist_track_ids = set(playlist_df["track_id"].dropna())

print(playlist_df.shape)
playlist_df.head()

## 3. Optional: Build a Candidate Pool from Spotify Search

A recommendation model can only recommend from songs it has available as candidates. This section searches Spotify for tracks related to the user's top artists and genres, enriches those tracks with artist metadata, and saves them to `data/candidate_tracks.csv`.

Set credentials in a local `.env` file before running this section:

```text
CLIENT_ID=your_client_id
CLIENT_SECRET=your_client_secret
```

The helper below also supports the clearer names `SPOTIFY_CLIENT_ID` and `SPOTIFY_CLIENT_SECRET`.

Spotify's Search API supports track search with field filters such as artist, year, and genre. The notebook uses search because the recommendation endpoint and genre seed endpoint are deprecated in the current Spotify Web API reference.

In [ ]:
def parse_list_like(value):
    if isinstance(value, list):
        return value
    if pd.isna(value) or value == "":
        return []
    try:
        parsed = ast.literal_eval(value)
        return parsed if isinstance(parsed, list) else []
    except (ValueError, SyntaxError):
        return [part.strip() for part in str(value).split(",") if part.strip()]


def find_env_file():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        env_path = candidate / ".env"
        if env_path.exists():
            return env_path
    return Path(".env")


def load_env_file(path=None):
    env_path = Path(path) if path is not None else find_env_file()
    if not env_path.exists():
        return

    for line in env_path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        os.environ.setdefault(key, value)

def get_spotify_token():
    load_env_file()
    client_id = os.getenv("SPOTIFY_CLIENT_ID") or os.getenv("CLIENT_ID")
    client_secret = os.getenv("SPOTIFY_CLIENT_SECRET") or os.getenv("CLIENT_SECRET")

    if not client_id or not client_secret:
        raise ValueError("Set CLIENT_ID and CLIENT_SECRET in .env before calling Spotify.")

    response = requests.post(
        "https://accounts.spotify.com/api/token",
        headers={"Content-Type": "application/x-www-form-urlencoded"},
        data={
            "grant_type": "client_credentials",
            "client_id": client_id,
            "client_secret": client_secret,
        },
        timeout=30,
    )
    response.raise_for_status()
    return response.json()["access_token"]


def spotify_get(url, token, params=None, sleep_time=0.05):
    response = requests.get(
        url,
        headers={"Authorization": f"Bearer {token}"},
        params=params,
        timeout=30,
    )

    if response.status_code == 429:
        retry_after = int(response.headers.get("Retry-After", "1"))
        time.sleep(retry_after)
        return spotify_get(url, token, params=params, sleep_time=sleep_time)

    response.raise_for_status()
    time.sleep(sleep_time)
    return response.json()


def track_to_row(track):
    artists = track.get("artists", [])
    album = track.get("album", {})
    artist_names = [artist.get("name") for artist in artists if artist.get("name")]
    artist_ids = [artist.get("id") for artist in artists if artist.get("id")]

    return {
        "track_id": track.get("id"),
        "track_name": track.get("name"),
        "artist_names": ", ".join(artist_names),
        "artist_ids": ", ".join(artist_ids),
        "main_artist_name": artist_names[0] if artist_names else None,
        "main_artist_id": artist_ids[0] if artist_ids else None,
        "album_id": album.get("id"),
        "album_name": album.get("name"),
        "release_date": album.get("release_date"),
        "duration_ms": track.get("duration_ms"),
        "duration_min": track.get("duration_ms") / 60000 if track.get("duration_ms") else np.nan,
        "popularity": track.get("popularity"),
        "explicit": track.get("explicit"),
        "track_url": track.get("external_urls", {}).get("spotify"),
        "is_local": track.get("is_local", False),
    }


def top_playlist_genres(df, n=12):
    genre_counter = Counter()
    for genres in df["main_artist_genres"].map(parse_list_like):
        genre_counter.update(genres)
    return [genre for genre, _ in genre_counter.most_common(n)]


def search_tracks(query, token, market="US", limit=50):
    data = spotify_get(
        "https://api.spotify.com/v1/search",
        token,
        params={"q": query, "type": "track", "market": market, "limit": limit},
    )
    return data.get("tracks", {}).get("items", [])


def get_artist_metadata(artist_ids, token):
    rows = []
    for artist_id in sorted(set(artist_ids)):
        if not artist_id:
            continue
        artist = spotify_get(f"https://api.spotify.com/v1/artists/{artist_id}", token)
        rows.append({
            "main_artist_id": artist.get("id"),
            "artist_name_from_api": artist.get("name"),
            "main_artist_genres": artist.get("genres", []),
            "artist_popularity": artist.get("popularity"),
            "artist_followers": artist.get("followers", {}).get("total"),
        })
    return pd.DataFrame(rows)


def build_candidate_pool_from_spotify(playlist_df, market="US", per_query_limit=50):
    token = get_spotify_token()

    top_artists = playlist_df["main_artist_name"].value_counts().head(20).index.tolist()
    top_genres = top_playlist_genres(playlist_df, n=20)

    queries = []
    queries.extend([f'artist:"{artist}"' for artist in top_artists])
    queries.extend([f'genre:"{genre}"' for genre in top_genres])

    rows = []
    for query in queries:
        tracks = search_tracks(query, token, market=market, limit=per_query_limit)
        rows.extend(track_to_row(track) for track in tracks)

    candidates = pd.DataFrame(rows)
    candidates = candidates.dropna(subset=["track_id"]).drop_duplicates("track_id")
    candidates = candidates[~candidates["track_id"].isin(playlist_track_ids)].copy()

    artist_metadata = get_artist_metadata(candidates["main_artist_id"].dropna().unique(), token)
    candidates = candidates.merge(artist_metadata, on="main_artist_id", how="left")
    candidates["source"] = "candidate"

    CANDIDATE_PATH.parent.mkdir(exist_ok=True)
    candidates.to_csv(CANDIDATE_PATH, index=False)
    return candidates


# Run this only when you need to create or refresh the candidate pool.
# candidate_df = build_candidate_pool_from_spotify(playlist_df, market="US", per_query_limit=50)
# candidate_df.shape

## 4. Load Candidate Tracks

The final recommendations come from this candidate pool after removing tracks already present in the user's playlist.

In [ ]:
if CANDIDATE_PATH.exists():
    candidate_df = pd.read_csv(CANDIDATE_PATH)
else:
    print("data/candidate_tracks.csv does not exist yet. Building it from Spotify Search...")
    candidate_df = build_candidate_pool_from_spotify(playlist_df, market="US", per_query_limit=50)

candidate_df["source"] = "candidate"
candidate_df = candidate_df.dropna(subset=["track_id"]).drop_duplicates("track_id")
candidate_df = candidate_df[~candidate_df["track_id"].isin(playlist_track_ids)].copy()

print(candidate_df.shape)
candidate_df.head()

## 5. Feature Engineering

The model uses metadata and artist-level features that are available for both playlist and candidate tracks:

- Numeric features: duration, track popularity, artist popularity, artist followers, release year.
- Binary/categorical features: explicit flag, title script, song title language.
- Multi-label features: artist genres and language-associated genre tags.

All features are fitted on the combined playlist + candidate set so both groups share the exact same columns.

In [ ]:
def extract_release_year(value):
    if pd.isna(value):
        return np.nan
    match = re.search(r"\d{4}", str(value))
    return int(match.group()) if match else np.nan


def build_feature_matrix(df, top_genres=None, top_tags=None):
    model_df = df.copy()

    model_df["release_year"] = model_df["release_date"].map(extract_release_year)
    model_df["explicit"] = model_df["explicit"].astype(str).str.lower().isin(["true", "1", "yes"])
    model_df["artist_followers_log"] = np.log1p(pd.to_numeric(model_df.get("artist_followers"), errors="coerce"))

    numeric_cols = ["duration_min", "popularity", "artist_popularity", "artist_followers_log", "release_year"]
    numeric_features = model_df[numeric_cols].apply(pd.to_numeric, errors="coerce")
    numeric_features = numeric_features.fillna(numeric_features.median())

    scaler = MinMaxScaler()
    numeric_features = pd.DataFrame(
        scaler.fit_transform(numeric_features),
        columns=numeric_cols,
        index=model_df.index,
    )

    categorical_cols = ["explicit", "title_script", "song_title_language"]
    for col in categorical_cols:
        if col not in model_df.columns:
            model_df[col] = "unknown"
    categorical_input = model_df[categorical_cols].fillna("unknown").astype(str)
    categorical_features = pd.get_dummies(categorical_input, prefix=categorical_cols)

    parsed_genres = model_df.get("main_artist_genres", pd.Series(index=model_df.index, dtype=object)).map(parse_list_like)
    parsed_tags = model_df.get("genre_language_tags", pd.Series(index=model_df.index, dtype=object)).map(parse_list_like)

    if top_genres is None:
        genre_counts = Counter(genre for genres in parsed_genres for genre in genres)
        top_genres = [genre for genre, _ in genre_counts.most_common(50)]

    if top_tags is None:
        tag_counts = Counter(tag for tags in parsed_tags for tag in tags)
        top_tags = [tag for tag, _ in tag_counts.most_common(20)]

    genre_features = pd.DataFrame(index=model_df.index)
    for genre in top_genres:
        genre_features[f"genre__{genre}"] = parsed_genres.map(lambda values, g=genre: int(g in values))

    tag_features = pd.DataFrame(index=model_df.index)
    for tag in top_tags:
        tag_features[f"tag__{tag}"] = parsed_tags.map(lambda values, t=tag: int(t in values))

    features = pd.concat([numeric_features, categorical_features, genre_features, tag_features], axis=1)
    return features, top_genres, top_tags


combined_df = pd.concat([playlist_df, candidate_df], ignore_index=True, sort=False)
features, top_genres, top_tags = build_feature_matrix(combined_df)

playlist_features = features.loc[combined_df["source"].eq("playlist")]
candidate_features = features.loc[combined_df["source"].eq("candidate")]

print(features.shape)
features.head()

## 6. Build Playlist Taste Profiles

The overall profile is the average feature vector across all songs in the user's playlist.

Weighted profiles can emphasize songs that are more recent additions or more popular. The default below keeps the baseline simple and transparent.

In [ ]:
overall_profile = playlist_features.mean(axis=0).to_frame().T

similarities = cosine_similarity(candidate_features, overall_profile).ravel()

recommendations = candidate_df.copy()
recommendations["similarity_to_playlist"] = similarities
recommendations = recommendations.sort_values("similarity_to_playlist", ascending=False)

recommendations[[
    "track_name",
    "artist_names",
    "album_name",
    "release_date",
    "popularity",
    "artist_popularity",
    "main_artist_genres",
    "similarity_to_playlist",
    "track_url",
]].head(25)

## 7. Recommendation Modes

The same feature matrix can support different recommendation experiences:

- `balanced`: most similar to the full playlist profile.
- `discovery`: similar to the playlist, but avoids artists already heavily represented in the playlist.
- `fresh`: similar to the playlist, with a small boost for newer releases.
- `popular`: similar to the playlist, with a small boost for track popularity.

In [ ]:
playlist_artists = set(playlist_df["main_artist_id"].dropna())


def normalize_series(series):
    values = pd.to_numeric(series, errors="coerce")
    if values.max() == values.min():
        return pd.Series(0.0, index=series.index)
    return (values - values.min()) / (values.max() - values.min())


def rank_recommendations(mode="balanced", n=30, drop_duplicate_artists=True):
    ranked = recommendations.copy()
    ranked["score"] = ranked["similarity_to_playlist"]

    if mode == "discovery":
        known_artist_penalty = ranked["main_artist_id"].isin(playlist_artists).astype(float) * 0.08
        ranked["score"] = ranked["score"] - known_artist_penalty
    elif mode == "fresh":
        ranked["release_year"] = ranked["release_date"].map(extract_release_year)
        ranked["score"] = ranked["score"] + 0.05 * normalize_series(ranked["release_year"])
    elif mode == "popular":
        ranked["score"] = ranked["score"] + 0.05 * normalize_series(ranked["popularity"])
    elif mode != "balanced":
        raise ValueError("mode must be one of: balanced, discovery, fresh, popular")

    display_cols = [
        "track_name",
        "artist_names",
        "album_name",
        "release_date",
        "popularity",
        "main_artist_genres",
        "similarity_to_playlist",
        "score",
        "track_url",
    ]
    return ranked.sort_values("score", ascending=False).head(n)[display_cols]


rank_recommendations(mode="balanced", n=20)

In [ ]:
rank_recommendations(mode="discovery")

## 8. Lightweight Validation

Recommendation quality is hard to measure without real user feedback. As a sanity check, this leave-one-out style evaluation asks whether a held-out playlist track looks similar to the rest of the playlist.

This does not prove the recommendations are good, but it checks whether the feature space captures the playlist's internal structure.

In [ ]:
def playlist_holdout_similarity(playlist_features, holdout_fraction=0.2, random_state=42):
    rng = np.random.default_rng(random_state)
    n_holdout = max(1, int(len(playlist_features) * holdout_fraction))
    holdout_idx = rng.choice(playlist_features.index.to_numpy(), size=n_holdout, replace=False)

    train_features = playlist_features.drop(index=holdout_idx)
    holdout_features = playlist_features.loc[holdout_idx]

    train_profile = train_features.mean(axis=0).to_frame().T
    holdout_similarity = cosine_similarity(holdout_features, train_profile).ravel()

    random_similarity = cosine_similarity(
        candidate_features.sample(n=min(n_holdout, len(candidate_features)), random_state=random_state),
        train_profile,
    ).ravel()

    return pd.DataFrame({
        "group": ["held_out_playlist"] * len(holdout_similarity) + ["random_candidate"] * len(random_similarity),
        "similarity": np.concatenate([holdout_similarity, random_similarity]),
    })


validation_df = playlist_holdout_similarity(playlist_features)
validation_df.groupby("group")["similarity"].describe()

## 9. Export Final Recommendations

Save a ranked recommendation table for later use in a report or app.

In [ ]:
final_recommendations = rank_recommendations(mode="balanced")
OUTPUT_PATH.parent.mkdir(exist_ok=True)
final_recommendations.to_csv(OUTPUT_PATH, index=False)

print(f"Saved recommendations to {OUTPUT_PATH}")
final_recommendations

## 10. Modelling Takeaways

- This is a content-based recommender, which fits the project because the data is centered on one user's playlist rather than many users' listening histories.
- The model recommends only from the candidate pool, so candidate generation is as important as the similarity model.
- The similarity score is interpretable because it is based on genre, language, artist metadata, release year, duration, and popularity.
- Future improvements could add user feedback, audio features if available, playlist section-specific profiles, or an app interface where the user can choose between balanced, discovery, fresh, and popular modes.